# visualization_002

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (136).ipynb`

| Field | Value |
|---|---|
| Section | `visualization` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 15 |
| Detected functions | close_position, close_position, trading_logic, walk_forward_optimization_quarterly |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import pandas_datareader as pdr
from datetime import datetime
import matplotlib.pyplot as plt

In [ ]:
data1 = pd.read_csv('ICICIBANK_minute_data_with_indicators.csv')
data2 = pd.read_csv('SBIN_minute_data_with_indicators.csv')

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline  

fig = plt.figure(figsize=(13, 7))
plt.plot(data1['close'], label='data1')
plt.plot(data2['close'], label='data2')
plt.legend()
plt.title("Stock Prices for data1 and data2")
plt.xlabel("Date")
plt.ylabel("close Price")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Create the figure
fig = plt.figure(figsize=(13, 7))

# Calculate the difference between the 'close' prices of data1.NS and data2.NS
price_difference = data1['close'] - data2['close']

# Plot the difference
plt.plot(price_difference, label='data1 - data2')
plt.legend()
plt.title("Difference in close Prices: data1 - data2")
plt.xlabel("Date")
plt.ylabel("Price Difference")
plt.show()


In [ ]:
data = pd.DataFrame({
    'datetime' : data1['date'],
    'data1': data1['close'],
    'data2': data2['close']
})
data

In [ ]:
import matplotlib.pyplot as plt

# Calculate the ratio of HdataC to ICICI
ratio = data['data1'] / data['data2']
ratio_mean = ratio.rolling(window=17).mean()
# Create the figure and plot the ratio
fig = plt.figure(figsize=(13, 7))
plt.plot(ratio, label='data1 / data2')

# Add a horizontal line at the mean of the ratio
plt.plot(ratio_mean, color='red', linestyle='--', label='Mean Ratio')

# Add legend, title, and labels
plt.legend()
plt.title("Ratio of data1 to data2 close Prices")
plt.xlabel("Date")
plt.ylabel("Ratio")
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Calculate the ratio of data1 to data2
ratio = data['data1'] / data['data2']

# Calculate the 43-min moving average
moving_average_43 = ratio.rolling(window=43).mean()

# Calculate the mean and standard deviation of the ratio
mean_ratio = ratio.mean()
std_ratio = ratio.std()

# Calculate the Z-score using the ratio
z_score = (ratio - moving_average_43) / moving_average_43.std()

# Create the figure and plot the Z-score
fig = plt.figure(figsize=(13, 7))
plt.plot(z_score, label='Z-Score of data1 / data2', color='purple')

# Add horizontal lines for Z-scores of -1 and 1 for reference
plt.axhline(0.02, color='green', linestyle='--', label='Z-Score = 0.02')
plt.axhline(-0.02, color='green', linestyle='--', label='Z-Score = -0.02')


# Add buy and sell markers
buy_signals = z_score[z_score < -0.02]
sell_signals = z_score[z_score > 0.02]

# Mark buy signals
plt.scatter(buy_signals.index, buy_signals, marker='^', color='green', label='Buy Signal', s=100)
# Mark sell signals
plt.scatter(sell_signals.index, sell_signals, marker='v', color='red', label='Sell Signal', s=100)

# Add legend, title, and labels
plt.legend()
plt.title("Z-Score of data1 to data2 Ratio with Buy/Sell Signals")
plt.xlabel("Date")
plt.ylabel("Z-Score")
plt.show()


In [ ]:
data['ratio'] = data['data1'] / data['data2']
data['difference'] = data['data1'] - data['data2']
data['z_score_pair'] = (ratio - moving_average_43) / moving_average_43.std()
data

In [ ]:
import pandas as pd

# Initialize positions
positions = pd.Series(index=data.index, data=0)

# Generate buy signals (when z-score is less than -0.05)
positions[data['z_score_pair'] < -0.02] = 1  # Buy position

# Generate sell signals (when z-score is greater than 0.05)
positions[data['z_score_pair'] > 0.02] = -1  # Sell position

# Close all positions (when z-score is equal to 0)
positions[data['z_score_pair'] == 0] = 0  # Close all positions

# Add the positions to the DataFrame
data['position'] = positions

# Display the resulting DataFrame
data

In [ ]:
import pandas as pd

# Convert the 'datetime' column to datetime format
data['datetime'] = pd.to_datetime(data['datetime'])

# Initialize parameters
initial_capital = 100000  # Starting capital
portfolio_value = initial_capital
current_position = 0
entry_price1, entry_price2 = None, None
entry_datetime = None
brokerage_fee = 0.0002  # 0.02% brokerage fee
risk_percentage = 1  # 100% risk per trade
tp_percentage = 1  # 100% take profit
sl_percentage = 1  # 100% stop loss
tradebook = []

# Function to handle trade closing and logging
def close_position(entry_price1, entry_price2, exit_price1, exit_price2, position_type, entry_datetime, exit_datetime, position_size1, position_size2):
    global portfolio_value
    # Calculate PnL based on position type
    pnl = (position_size1 * (exit_price1 - entry_price1) + position_size2 * (entry_price2 - exit_price2)
           if position_type == 1 else
           position_size1 * (entry_price1 - exit_price1) + position_size2 * (exit_price2 - entry_price2))
    
    # Apply brokerage fees
    transaction_cost = brokerage_fee * (position_size1 * (entry_price1 + exit_price1) + position_size2 * (entry_price2 + exit_price2))
    pnl -= transaction_cost
    portfolio_value += pnl

    # Log trade details
    tradebook.append({
        'Entry Price Stock1': entry_price1,
        'Entry Price Stock2': entry_price2,
        'Exit Price Stock1': exit_price1,
        'Exit Price Stock2': exit_price2,
        'Trade Type': 'Short' if position_type == -1 else 'Long',
        'Position Size Stock1': position_size1,
        'Position Size Stock2': position_size2,
        'PnL': round(pnl, 2),
        'Transaction Cost': round(transaction_cost, 2),
        'Portfolio Value': round(portfolio_value, 2),
        'Entry Date': entry_datetime.date(),
        'Entry Time': entry_datetime.time(),
        'Exit Date': exit_datetime.date(),
        'Exit Time': exit_datetime.time()
    })

# Iterate over the dataset rows
for i, row in data.iterrows():
    position = row['position']
    price1 = row['data1']
    price2 = row['data2']
    current_datetime = row['datetime']  # Assuming 'datetime' is the column name in your DataFrame

    # Skip iteration if price1 or price2 is NaN
    if pd.isna(price1) or pd.isna(price2):
        continue

    # Maximum allowable risk in dollars
    max_risk = portfolio_value * risk_percentage

    # Skip iteration if price difference is zero to avoid division errors
    price_diff = abs(price1 - price2)
    if price_diff == 0:
        continue

    # Calculate position sizes for stock1 and stock2 based on max risk
    position_size1 = round(max_risk / (2 * price1)) if price1 > 0 else 0
    position_size2 = round(max_risk / (2 * price2)) if price2 > 0 else 0 

    # Take Profit and Stop Loss check
    if current_position != 0:
        pnl_percentage = ((price1 - entry_price1) / entry_price1 if current_position == 1 else (entry_price1 - price1) / entry_price1) \
                         + ((price2 - entry_price2) / entry_price2 if current_position == -1 else (entry_price2 - price2) / entry_price2)

        # Check if TP or SL is reached
        if pnl_percentage >= tp_percentage:
            close_position(entry_price1, entry_price2, price1, price2, current_position, entry_datetime, current_datetime, position_size1, position_size2)
            current_position = 0
            continue  # Move to next row after closing position
        elif pnl_percentage <= -sl_percentage:
            close_position(entry_price1, entry_price2, price1, price2, current_position, entry_datetime, current_datetime, position_size1, position_size2)
            current_position = 0
            continue  # Move to next row after closing position

    # Open a new Buy position
    if position == 1 and current_position != 1:
        if current_position != 0:  # Close any existing position first
            close_position(entry_price1, entry_price2, price1, price2, current_position, entry_datetime, current_datetime, position_size1, position_size2)
        
        # Open the new buy position
        entry_price1, entry_price2 = price1, price2
        entry_datetime = current_datetime  # Capture entry time
        current_position = 1

    # Open a new Sell position
    elif position == -1 and current_position != -1:
        if current_position != 0:  # Close any existing position first
            close_position(entry_price1, entry_price2, price1, price2, current_position, entry_datetime, current_datetime, position_size1, position_size2)
        
        # Open the new sell position
        entry_price1, entry_price2 = price1, price2
        entry_datetime = current_datetime  # Capture entry time
        current_position = -1

    # Close position if `position` is set to 0
    elif position == 0 and current_position != 0:
        close_position(entry_price1, entry_price2, price1, price2, current_position, entry_datetime, current_datetime, position_size1, position_size2)
        
        # Reset position to neutral
        current_position = 0
        entry_price1, entry_price2 = None, None
        entry_datetime = None

# Convert tradebook to DataFrame and export to CSV
tradebook_df = pd.DataFrame(tradebook)
tradebook_df.to_csv('tradebook.csv', index=False)

# Display the tradebook
print(tradebook_df)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
from matplotlib.backends.backend_pdf import PdfPages

# Load the tradebook data
tradebook_df = pd.read_csv('tradebook.csv')

# Convert 'Entry Date' and 'Exit Date' to datetime format if not already done
tradebook_df['Entry DateTime'] = pd.to_datetime(tradebook_df['Entry Date'] + ' ' + tradebook_df['Entry Time'])
tradebook_df['Exit DateTime'] = pd.to_datetime(tradebook_df['Exit Date'] + ' ' + tradebook_df['Exit Time'])

# Create a PDF file to save all plots
with PdfPages('trading_analysis_plots.pdf') as pdf:

    # 1. Plot Portfolio Value with Drawdown
    tradebook_df['Max Portfolio Value'] = tradebook_df['Portfolio Value'].cummax()
    tradebook_df['Drawdown'] = (tradebook_df['Portfolio Value'] - tradebook_df['Max Portfolio Value']) / tradebook_df['Max Portfolio Value']
    
    plt.figure(figsize=(12, 6))
    plt.plot(tradebook_df['Portfolio Value'], label='Portfolio Value', color='blue')
    plt.fill_between(tradebook_df.index, 
                     tradebook_df['Portfolio Value'], 
                     tradebook_df['Max Portfolio Value'], 
                     where=tradebook_df['Drawdown'] > 0, 
                     color='red', alpha=0.3, label='Drawdown')
    plt.title('Portfolio Value with Drawdown')
    plt.xlabel('Trade Number')
    plt.ylabel('Portfolio Value')
    plt.legend()
    plt.grid()
    pdf.savefig()  # Save the plot to the PDF
    plt.close()

    # 2. Plot Drawdown Over Time
    plt.figure(figsize=(12, 6))
    plt.plot(tradebook_df['Drawdown'], color='red', label='Drawdown')
    plt.fill_between(tradebook_df.index, tradebook_df['Drawdown'], color='red', alpha=0.3)
    plt.title('Portfolio Drawdown Over Time')
    plt.xlabel('Trade Number')
    plt.ylabel('Drawdown (%)')
    plt.legend()
    plt.grid()
    pdf.savefig()  # Save the plot to the PDF
    plt.close()

    # 3. Plot Monthly Profit Percentage Heatmap
    tradebook_df['Year'] = tradebook_df['Entry DateTime'].dt.year
    tradebook_df['Month'] = tradebook_df['Entry DateTime'].dt.month
    first_trade_of_month = tradebook_df.groupby(['Year', 'Month']).first().reset_index()
    tradebook_df = tradebook_df.merge(first_trade_of_month[['Year', 'Month', 'Portfolio Value']], 
                                       on=['Year', 'Month'], suffixes=('', '_FirstTrade'))
    tradebook_df['Monthly Profit %'] = (tradebook_df['PnL'] / tradebook_df['Portfolio Value_FirstTrade']) * 100
    monthly_profit_df = tradebook_df.groupby(['Year', 'Month'])['Monthly Profit %'].sum().reset_index()
    heatmap_data = monthly_profit_df.pivot_table(values='Monthly Profit %', index='Year', columns='Month', aggfunc='sum')
    heatmap_data.fillna(0, inplace=True)
    
    cmap = mcolors.ListedColormap(['#ff6666', '#3BB143'])  # Red for negative, Green for positive
    norm = mcolors.TwoSlopeNorm(vmin=heatmap_data.min().min(), vcenter=0, vmax=heatmap_data.max().max())
    
    plt.figure(figsize=(14, 6))
    sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap=cmap, norm=norm, cbar_kws={'label': 'Monthly Profit %'}, linewidths=0.5)
    plt.title("Monthly Profit Percentage Heatmap")
    plt.xlabel("Month")
    plt.ylabel("Year")
    plt.xticks(ticks=range(1, 13), labels=["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
    plt.tight_layout()
    pdf.savefig()  # Save the plot to the PDF
    plt.close()

    # 4. Plot Total PnL by Weekday
    tradebook_df['Weekday'] = tradebook_df['Entry DateTime'].dt.day_name()
    weekday_pnl = tradebook_df.groupby('Weekday')['PnL'].sum()
    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
    weekday_pnl = weekday_pnl.reindex(weekday_order)
    
    plt.figure(figsize=(13, 5))
    plt.bar(weekday_pnl.index, weekday_pnl.values, color='skyblue')
    plt.xlabel("Weekday")
    plt.ylabel("Total PnL")
    plt.title("Total PnL by Weekday")
    plt.xticks(rotation=45)
    pdf.savefig()  # Save the plot to the PDF
    plt.close()

    # 5. Plot PnL Percentage Distribution (Confusion Matrix Style)
    tradebook_df['PnL Percentage'] = (tradebook_df['PnL'] / tradebook_df['Portfolio Value']) * 100
    bins = [-np.inf, -7, -6, -5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5, 6, 7, np.inf]
    labels = ['<-7%', '-7% to -6%', '-6% to -5%', '-5% to -4%', '-4% to -3%', '-3% to -2%', '-2% to -1%',
              '-1% to 0%', '0% to 1%', '1% to 2%', '2% to 3%', '3% to 4%', '4% to 5%', '5% to 6%', '6% to 7%', '>7%']
    tradebook_df['PnL Category'] = pd.cut(tradebook_df['PnL Percentage'], bins=bins, labels=labels)
    pnl_category_counts = pd.crosstab(index=tradebook_df['PnL Category'], columns='Count')
    
    plt.figure(figsize=(13, 5))
    sns.heatmap(pnl_category_counts.T, annot=True, cmap='Blues', fmt='g', cbar=False)
    plt.title("PnL Percentage Distribution (Confusion Matrix Style)")
    plt.ylabel("PnL Category")
    plt.xlabel("Frequency")
    pdf.savefig()  # Save the plot to the PDF
    plt.close()

    # 6. Plot Yearly Returns
    yearly_returns = tradebook_df.groupby('Year')['PnL'].sum()
    yearly_returns_percent = (yearly_returns / tradebook_df.groupby('Year')['Portfolio Value'].first()) * 100
    
    # Set y-axis limit to avoid the 3500% outlier from skewing the plot
    max_y_value = yearly_returns_percent[yearly_returns_percent < 1000].max() * 1.1  # 10% more than the maximum return under 1000%
    
    plt.figure(figsize=(12, 6))
    plt.bar(yearly_returns_percent.index, yearly_returns_percent.values, color='skyblue', edgecolor='black')
    plt.title("Yearly Returns")
    plt.xlabel("Year")
    plt.ylabel("Profit Percentage (%)")
    plt.xticks(yearly_returns_percent.index, rotation=45)
    plt.ylim(0, max_y_value)  # Set the y-axis limit
    plt.grid(True)
    
    # Adding annotation for 2022 outlier
    if 2022 in yearly_returns_percent.index and yearly_returns_percent[2022] > 1000:
        plt.annotate(f'2022: {yearly_returns_percent[2022]:.2f}%', 
                     xy=(2022, yearly_returns_percent[2022]), 
                     xytext=(2022, max_y_value * 0.9),
                     arrowprops=dict(facecolor='red', arrowstyle='->'),
                     fontsize=10, color='red')

    pdf.savefig()  # Save the plot to the PDF
    plt.close()
    

In [ ]:
import pandas as pd
from datetime import timedelta

# Load data and ensure 'datetime' is timezone-aware
#if data['datetime'].dtype == 'datetime64[ns, Asia/Kolkata]':
    #data['datetime'] = pd.to_datetime(data['datetime']).dt.tz_convert('Asia/Kolkata')
#else:
    #data['datetime'] = pd.to_datetime(data['datetime']).dt.tz_localize('Asia/Kolkata')

# Initialize parameters
initial_capital = 100000  # Starting capital
brokerage_fee = 0.0002  # 0.02% brokerage fee
tp_percentage = 1.0  # 100% take profit
sl_percentage = 1.0  # 100% stop loss
risk_percentage = 1.0  # 100% risk per trade

# Trade logging function
def close_position(entry_price1, entry_price2, exit_price1, exit_price2, position_type, entry_datetime, exit_datetime, position_size1, position_size2):
    global portfolio_value
    # Calculate PnL based on position type
    pnl = (position_size1 * (exit_price1 - entry_price1) + position_size2 * (entry_price2 - exit_price2)
           if position_type == 1 else
           position_size1 * (entry_price1 - exit_price1) + position_size2 * (exit_price2 - entry_price2))
    
    # Apply brokerage fees
    transaction_cost = brokerage_fee * (position_size1 * (entry_price1 + exit_price1) + position_size2 * (entry_price2 + exit_price2))
    pnl -= transaction_cost
    portfolio_value += pnl

    # Log trade details
    tradebook.append({
        'Entry Price Stock1': entry_price1,
        'Entry Price Stock2': entry_price2,
        'Exit Price Stock1': exit_price1,
        'Exit Price Stock2': exit_price2,
        'Trade Type': 'Short' if position_type == -1 else 'Long',
        'Position Size Stock1': position_size1,
        'Position Size Stock2': position_size2,
        'PnL': round(pnl, 2),
        'Transaction Cost': round(transaction_cost, 2),
        'Portfolio Value': round(portfolio_value, 2),
        'Entry Date': entry_datetime.date(),
        'Entry Time': entry_datetime.time(),
        'Exit Date': exit_datetime.date(),
        'Exit Time': exit_datetime.time()
    })

# Placeholder trading logic function
def trading_logic(data, initial_capital):
    global portfolio_value, tradebook
    portfolio_value = initial_capital
    current_position = 0
    entry_price1, entry_price2 = None, None
    entry_datetime = None
    tradebook = []

    for i, row in data.iterrows():
        position = row['position']
        price1 = row['data1']
        price2 = row['data2']
        current_datetime = row['datetime']

        # Skip iteration if price1 or price2 is NaN
        if pd.isna(price1) or pd.isna(price2):
            continue

        # Maximum allowable risk in dollars
        max_risk = portfolio_value * risk_percentage

        # Calculate position sizes for stock1 and stock2 based on max risk
        position_size1 = round(max_risk / (2 * price1)) if price1 > 0 else 0
        position_size2 = round(max_risk / (2 * price2)) if price2 > 0 else 0 

        # Take Profit and Stop Loss check
        if current_position != 0:
            pnl_percentage = ((price1 - entry_price1) / entry_price1 if current_position == 1 else (entry_price1 - price1) / entry_price1) \
                             + ((price2 - entry_price2) / entry_price2 if current_position == -1 else (entry_price2 - price2) / entry_price2)

            # Check if TP or SL is reached
            if pnl_percentage >= tp_percentage:
                close_position(entry_price1, entry_price2, price1, price2, current_position, entry_datetime, current_datetime, position_size1, position_size2)
                current_position = 0
                continue
            elif pnl_percentage <= -sl_percentage:
                close_position(entry_price1, entry_price2, price1, price2, current_position, entry_datetime, current_datetime, position_size1, position_size2)
                current_position = 0
                continue

        # Open a new Buy position
        if position == 1 and current_position != 1:
            if current_position != 0:  # Close any existing position first
                close_position(entry_price1, entry_price2, price1, price2, current_position, entry_datetime, current_datetime, position_size1, position_size2)
            
            # Open the new buy position
            entry_price1, entry_price2 = price1, price2
            entry_datetime = current_datetime  # Capture entry time
            current_position = 1

        # Open a new Sell position
        elif position == -1 and current_position != -1:
            if current_position != 0:  # Close any existing position first
                close_position(entry_price1, entry_price2, price1, price2, current_position, entry_datetime, current_datetime, position_size1, position_size2)
            
            # Open the new sell position
            entry_price1, entry_price2 = price1, price2
            entry_datetime = current_datetime  # Capture entry time
            current_position = -1

        # Close position if `position` is set to 0
        elif position == 0 and current_position != 0:
            close_position(entry_price1, entry_price2, price1, price2, current_position, entry_datetime, current_datetime, position_size1, position_size2)
            
            # Reset position to neutral
            current_position = 0
            entry_price1, entry_price2 = None, None
            entry_datetime = None

    return {
        'Portfolio Value': portfolio_value,
        'PnL': portfolio_value - initial_capital,
        'Trade Count': len(tradebook)
    }

# Walk-Forward Optimization function
def walk_forward_optimization_quarterly(data):
    performance_results = []
    train_period = timedelta(days=360)  # Train on 1 year of data
    test_period = timedelta(days=90)    # Test on 3 months of data

    # Start and end dates for the sliding window
    current_start = data['datetime'].iloc[0]
    end_date = data['datetime'].max()

    while current_start + train_period + test_period <= end_date:
        # Define training and testing periods
        train_end = current_start + train_period
        test_end = train_end + test_period

        # Split the data for training and testing
        train_data = data[(data['datetime'] >= current_start) & (data['datetime'] < train_end)]
        test_data = data[(data['datetime'] >= train_end) & (data['datetime'] < test_end)]

        # Run the trading strategy on test data
        result = trading_logic(test_data, initial_capital)
        result['Train Period Start'] = current_start
        result['Train Period End'] = train_end
        result['Test Period Start'] = train_end
        result['Test Period End'] = test_end

        # Store the results
        performance_results.append(result)

        # Shift the start date forward by test period (3 months)
        current_start += test_period

    return pd.DataFrame(performance_results)

# Run walk-forward optimization
performance_df = walk_forward_optimization_quarterly(data)

# Save results to CSV
performance_df.to_csv('quarterly_walk_forward_performance.csv', index=False)
print(performance_df)


In [ ]:
import matplotlib.pyplot as plt

# Assuming performance_df contains the results from walk-forward optimization
# Calculate PnL percentage (PnL / initial capital * 100)
performance_df['PnL Percentage'] = (performance_df['PnL'] / initial_capital) * 100

# Plot the histogram of PnL Percentage values vs Test Periods
plt.figure(figsize=(14, 7))

# Plot the histogram of PnL Percentage values
plt.bar(performance_df['Test Period Start'].astype(str), performance_df['PnL Percentage'], edgecolor='black')
plt.axhline(15, color='green', linestyle='--', label='Target PnL Percentage (15%)')
plt.axhline(25, color='blue', linestyle='--', label='Target PnL Percentage (25%)')
plt.axhline(50, color='purple', linestyle='--', label='Target PnL Percentage (50%)')

# Add labels, legend, and title
plt.xlabel('Test Period')
plt.ylabel('PnL Percentage')
plt.title('Distribution of PnL Percentage across Test Periods')
plt.legend()

# Rotate x-axis labels for better readability
plt.xticks(rotation=45)

# Display the plot
plt.tight_layout()
plt.show()


In [ ]:
performance_df['PnL Percentage'].mean()